In [1]:
import pandas as pd
import numpy as np
import shap
from xgboost import XGBClassifier
from scipy.stats import mannwhitneyu

In [3]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install scipy.stats

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scipy.stats (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for scipy.stats


In [4]:
df = pd.read_csv("churnData_Fixed.csv")
 
leak_cols = ["signup_date", "year_month", "window_id"]
model_features = [c for c in df.columns if c not in leak_cols + ["Churn"]]
 
N_WINDOWS = df["window_id"].nunique()
print(f"Loaded {df.shape[0]} rows across {N_WINDOWS} windows, {len(model_features)} features")

Loaded 7043 rows across 10 windows, 32 features


In [5]:
window0 = df[df["window_id"] == 0]
X_window0 = window0[model_features]
y_window0 = window0["Churn"]
 
neg_count = (y_window0 == 0).sum()
pos_count = (y_window0 == 1).sum()
scale_pos_weight = neg_count / pos_count
 
shap_base_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight,
)
shap_base_model.fit(X_window0, y_window0)
 
print(f"\nBaseline SHAP model trained on window 0 ({len(X_window0)} customers, "
      f"scale_pos_weight={scale_pos_weight:.2f})")


Baseline SHAP model trained on window 0 (705 customers, scale_pos_weight=23.31)


In [6]:
explainer = shap.TreeExplainer(shap_base_model)
 
shap_values_by_window = {}
for w in sorted(df["window_id"].unique()):
    window_data = df[df["window_id"] == w][model_features]
    sv = explainer.shap_values(window_data)
    shap_values_by_window[w] = pd.DataFrame(sv, columns=model_features)
    print(f"Window {w}: computed SHAP values for {len(window_data)} customers")

Window 0: computed SHAP values for 705 customers
Window 1: computed SHAP values for 704 customers
Window 2: computed SHAP values for 704 customers
Window 3: computed SHAP values for 704 customers
Window 4: computed SHAP values for 705 customers
Window 5: computed SHAP values for 704 customers
Window 6: computed SHAP values for 704 customers
Window 7: computed SHAP values for 704 customers
Window 8: computed SHAP values for 704 customers
Window 9: computed SHAP values for 705 customers


In [7]:
ALPHA = 0.05  # significance threshold — standard convention, not tuned
 
def rank_biserial_effect_size(n1, n2, u_stat):
    """
    Effect size derived from the same U statistic: ranges -1 to 1, measures
    HOW LARGE a distributional shift is. Needed alongside the p-value
    because p-values collapse to ~0 for any large-enough shift once sample
    sizes are in the hundreds, making same-magnitude comparisons impossible
    by p-value alone (several features can all show p=0.0).
    """
    return 1 - (2 * u_stat) / (n1 * n2)
 
baseline_shap = shap_values_by_window[0]
 
drift_results = []
for w in sorted(df["window_id"].unique()):
    if w == 0:
        continue
    window_shap = shap_values_by_window[w]
    for feature in model_features:
        stat, p_value = mannwhitneyu(
            baseline_shap[feature], window_shap[feature], alternative="two-sided"
        )
        effect_size = rank_biserial_effect_size(len(baseline_shap), len(window_shap), stat)
        drift_results.append({
            "window_id": w,
            "feature": feature,
            "u_statistic": stat,
            "p_value": p_value,
            "effect_size": abs(effect_size),
            "significant_drift": p_value < ALPHA,
        })
 
drift_df = pd.DataFrame(drift_results)
print(f"\nRan {len(drift_df)} Mann-Whitney tests "
      f"({N_WINDOWS - 1} windows x {len(model_features)} features)")


Ran 288 Mann-Whitney tests (9 windows x 32 features)


In [8]:
print("\n--- Number of features with significant drift, per window ---")
print(drift_df.groupby("window_id")["significant_drift"].sum())
 
print("\n--- Features most often flagged as drifting, across all windows ---")
feature_drift_counts = drift_df.groupby("feature")["significant_drift"].sum().sort_values(ascending=False)
print(feature_drift_counts.head(10))
 
print("\n--- Features with the LARGEST average effect size (magnitude of shift) ---")
feature_effect_avg = drift_df.groupby("feature")["effect_size"].mean().sort_values(ascending=False)
print(feature_effect_avg.head(10))
 
drift_df.to_csv("phase3_drift_results.csv", index=False)
print("\nFull per-window, per-feature results saved to phase3_drift_results.csv")


--- Number of features with significant drift, per window ---
window_id
1    15
2    19
3    19
4    18
5    18
6    19
7    20
8    20
9    20
Name: significant_drift, dtype: int64

--- Features most often flagged as drifting, across all windows ---
feature
Contract_Two year                        9
PaperlessBilling                         9
OnlineSecurity_Yes                       9
cpi_mom_pct_change                       9
tenure                                   9
StreamingMovies_Yes                      9
PaymentMethod_Mailed check               9
PaymentMethod_Credit card (automatic)    9
StreamingTV_Yes                          9
TechSupport_Yes                          9
Name: significant_drift, dtype: int64

--- Features with the LARGEST average effect size (magnitude of shift) ---
feature
tenure                                   0.896677
Contract_Two year                        0.704984
cpi_mom_pct_change                       0.558738
Contract_One year                     

In [9]:
INJECT_FEATURE = "MonthlyCharges"
INJECT_SHIFT_STD = 2.0     # shift by 2 standard deviations — large and
                            # unambiguous, easy to confirm was detected
 
injected_window_data = X_window0.copy()  # identical copy of window 0's own data
 
feature_std = df[INJECT_FEATURE].std()
injected_window_data[INJECT_FEATURE] = injected_window_data[INJECT_FEATURE] + (INJECT_SHIFT_STD * feature_std)
 
print(f"\nInjected a {INJECT_SHIFT_STD}-std-dev shift into '{INJECT_FEATURE}' "
      f"on a copy of window 0's own data (isolates the test from real-window confounds)")
 
sv_injected = explainer.shap_values(injected_window_data)
injected_shap = pd.DataFrame(sv_injected, columns=model_features)
 
injection_results = []
for feature in model_features:
    stat, p_value = mannwhitneyu(
        baseline_shap[feature], injected_shap[feature], alternative="two-sided"
    )
    effect_size = rank_biserial_effect_size(len(baseline_shap), len(injected_shap), stat)
    injection_results.append({
        "feature": feature,
        "p_value": p_value,
        "effect_size": abs(effect_size),
        "significant_drift": p_value < ALPHA,
    })
 
# Sort by effect size (magnitude of shift), not just p-value, so ties at
# p=0.0 are broken by which feature actually shifted the most
injection_df = pd.DataFrame(injection_results).sort_values("effect_size", ascending=False).reset_index(drop=True)
 
print("\n--- Synthetic injection test results (top 10 by effect size) ---")
print(injection_df.head(10))
 
detected = injection_df.iloc[0]["feature"] == INJECT_FEATURE
print(f"\nDetector correctly identified '{INJECT_FEATURE}' as the top drifting feature: {detected}")
 
 
print("\nPhase 3 complete.")
print("`shap_base_model`       -> frozen model used for all SHAP explanations (never retrained)")
print("`shap_values_by_window` -> dict of per-window SHAP value DataFrames")
print("`drift_df`              -> full Mann-Whitney results, real windows (also saved to CSV)")
print("`injection_df`          -> validation: does the detector find a KNOWN injected shift")


Injected a 2.0-std-dev shift into 'MonthlyCharges' on a copy of window 0's own data (isolates the test from real-window confounds)

--- Synthetic injection test results (top 10 by effect size) ---
                          feature       p_value  effect_size  \
0                  MonthlyCharges  4.487912e-45     0.433383   
1              cpi_mom_pct_change  1.706910e-43     0.425176   
2               Contract_Two year  4.076809e-33     0.368794   
3  PaymentMethod_Electronic check  2.901278e-29     0.345459   
4               Contract_One year  8.133281e-27     0.329790   
5                    PhoneService  8.768227e-16     0.236570   
6                OnlineBackup_Yes  1.671241e-06     0.147339   
7                   SeniorCitizen  6.537738e-04     0.104846   
8      PaymentMethod_Mailed check  7.830192e-04     0.103323   
9                    TotalCharges  8.801324e-04     0.102325   

   significant_drift  
0               True  
1               True  
2               True  
3    